# 🏠 House Price Prediction — Feature Engineering & Modeling



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, OrdinalEncoder




In [2]:
df = pd.read_csv("AmesHousing.csv")

df.columns = df.columns.str.replace(" ", "_").str.replace("/", "_")

print(f"Dataset shape: {df.shape}")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'AmesHousing.csv'

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 3. 🧩 Handling Missing Values



In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct.round(2)})
print(f"Columns with missing values: {len(missing_df)}")
missing_df


In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(x=missing_pct.values, y=missing_pct.index, palette="rocket")
plt.xlabel("% Missing")
plt.title("Missing Values by Column")
plt.tight_layout()
plt.show()


In [ ]:
none_cols = [
    "Pool_QC", "Misc_Feature", "Alley", "Fence", "Fireplace_Qu",
    "Garage_Type", "Garage_Finish", "Garage_Qual", "Garage_Cond",
    "Bsmt_Qual", "Bsmt_Cond", "Bsmt_Exposure", "BsmtFin_Type_1", "BsmtFin_Type_2",
    "Mas_Vnr_Type"
]
for c in none_cols:
    if c in df.columns:
        df[c] = df[c].fillna("None")

num_cols_missing = df.select_dtypes(include=[np.number]).columns[df.select_dtypes(include=[np.number]).isnull().any()]
for c in num_cols_missing:
    df[c] = df[c].fillna(df[c].median())

cat_cols_missing = df.select_dtypes(include="object").columns[df.select_dtypes(include="object").isnull().any()]
for c in cat_cols_missing:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())


 📈 Log Transformation on Skewed Features


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title(f"SalePrice (skew={df['SalePrice'].skew():.2f})")

df["SalePrice_Log"] = np.log1p(df["SalePrice"])
sns.histplot(df["SalePrice_Log"], kde=True, ax=axes[0, 1], color="seagreen")
axes[0, 1].set_title(f"SalePrice_Log (skew={df['SalePrice_Log'].skew():.2f})")

sns.histplot(df["Lot_Area"], kde=True, ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title(f"Lot_Area (skew={df['Lot_Area'].skew():.2f})")

df["Lot_Area_Log"] = np.log1p(df["Lot_Area"])
sns.histplot(df["Lot_Area_Log"], kde=True, ax=axes[1, 1], color="seagreen")
axes[1, 1].set_title(f"Lot_Area_Log (skew={df['Lot_Area_Log'].skew():.2f})")

plt.tight_layout()
plt.show()


## 5. 🛠️ Feature Creation





In [ ]:
df["HouseAge"] = df["Yr_Sold"] - df["Year_Built"]
df["HouseAge"] = df["HouseAge"].clip(lower=0)  

df["TotalBathrooms"] = (
    df["Full_Bath"] + 0.5 * df["Half_Bath"] +
    df["Bsmt_Full_Bath"] + 0.5 * df["Bsmt_Half_Bath"]
)


porch_cols = ["Open_Porch_SF", "Enclosed_Porch", "3Ssn_Porch", "Screen_Porch", "Wood_Deck_SF"]
df["TotalPorchArea"] = df[porch_cols].sum(axis=1)

df[["HouseAge", "TotalBathrooms", "TotalPorchArea"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["HouseAge"], kde=True, ax=axes[0], color="coral")
axes[0].set_title("HouseAge")
sns.histplot(df["TotalBathrooms"], kde=True, ax=axes[1], color="coral")
axes[1].set_title("TotalBathrooms")
sns.histplot(df["TotalPorchArea"], kde=True, ax=axes[2], color="coral")
axes[2].set_title("TotalPorchArea")
plt.tight_layout()
plt.show()


## 6. 🔢 Ordinal Encoding


In [ ]:
quality_map = {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}

for col in ["Exter_Qual", "Kitchen_Qual"]:
    df[col + "_Encoded"] = df[col].map(quality_map)

df[["Exter_Qual", "Exter_Qual_Encoded", "Kitchen_Qual", "Kitchen_Qual_Encoded"]].head()


## 7. 🎯 One-Hot Encoding


|

In [ ]:
print("Unique Neighborhoods:", df["Neighborhood"].nunique())
print("Unique House Styles:", df["House_Style"].nunique())

df_encoded = pd.get_dummies(df, columns=["Neighborhood", "House_Style"], prefix=["Nbhd", "Style"], drop_first=True)
print(f"\nShape before one-hot encoding: {df.shape}")
print(f"Shape after one-hot encoding: {df_encoded.shape}")


## 8. 🚨 Outlier Detection


In [ ]:
outlier_check_cols = ["SalePrice", "Gr_Liv_Area", "Lot_Area", "TotalBathrooms", "HouseAge"]

z_outliers = pd.DataFrame(index=df.index)
for col in outlier_check_cols:
    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers[col] = z_scores > 3

print("Outliers detected via Z-Score (threshold=3):")
print(z_outliers.sum())


In [ ]:
iqr_outliers = pd.DataFrame(index=df.index)
for col in outlier_check_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_outliers[col] = (df[col] < lower_bound) | (df[col] > upper_bound)

print("Outliers detected via IQR method:")
print(iqr_outliers.sum())


In [ ]:
fig, axes = plt.subplots(1, len(outlier_check_cols), figsize=(20, 5))
for i, col in enumerate(outlier_check_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
combined_outliers = (z_outliers.any(axis=1)) & (iqr_outliers.any(axis=1))
print(f"Rows flagged as outliers by BOTH Z-score and IQR: {combined_outliers.sum()} out of {len(df)}")

df_clean = df_encoded[~combined_outliers].reset_index(drop=True)
print(f"Shape after outlier removal: {df_clean.shape}")


## 9. ⚖️ Feature Scaling (StandardScaler)


In [ ]:
numeric_features_to_scale = [
    "Lot_Area_Log", "Gr_Liv_Area", "Total_Bsmt_SF", "HouseAge",
    "TotalBathrooms", "TotalPorchArea", "Overall_Qual", "Overall_Cond",
    "Exter_Qual_Encoded", "Kitchen_Qual_Encoded", "Garage_Area"
]
numeric_features_to_scale = [c for c in numeric_features_to_scale if c in df_clean.columns]

scaler = StandardScaler()
df_scaled = df_clean.copy()
df_scaled[numeric_features_to_scale] = scaler.fit_transform(df_clean[numeric_features_to_scale])

df_scaled[numeric_features_to_scale].describe().T[["mean", "std"]]
